Jupyter notebook to run through the getting started section of the icet Python library (https://icet.materialsmodeling.org/)

## Constructing the cluster expansion, Ag-Pd

In [4]:
from ase.db import connect
from icet import ClusterSpace, StructureContainer, ClusterExpansion
from trainstation import CrossValidationEstimator

In [ ]:
db = connect('../data/reference_data.db')
primitive_structure = db.get(id=1).toatoms()  # primitive structure

In order to create the cluster expansion, we must create a ClusterSpace based on a prototype structure. When initiating the former, one must also provide cutoffs and the chemical elements that are allowed to occupy different lattice sites. The cut-offs define the longest distance between atom clusters in Angstroms, or Å.

For more details here please consult https://icet.materialsmodeling.org/get_started/construct_cluster_expansion.html

In [7]:
cs = ClusterSpace(structure=primitive_structure,
                  cutoffs=[13.5, 6.5, 6.0], # the cut-offs in question, 13.5 for pairs, 6.5 for triplets, 6 for quadruplets.
                  chemical_symbols=['Ag', 'Pd']) # the allowed elements
display(cs)

Field,Value
Space group,Fm-3m (225)
Sublattice A,"('Ag', 'Pd')"
Cutoffs,"[13.5, 6.5, 6.0]"
Total number of parameters,82
Number of parameters of order 0,1
Number of parameters of order 1,1
Number of parameters of order 2,25
Number of parameters of order 3,20
Number of parameters of order 4,35
fractional_position_tolerance,2e-06


We now need to initiate a StructureContainer. Only adding one property here, but in theory can add many.

For clarity here, ClusterSpace vs StructureContainer:

 ClusterSpace      = the basis. Given a prototype lattice, cutoffs, and allowed
                     elements, it enumerates all symmetrically distinct clusters
                     (pairs, triplets, ...). Defines *what* to measure, no data yet.

 StructureContainer = the data. For each structure in the DB, projects it onto
                      the ClusterSpace basis -> feature vector, paired with target
                      property (e.g. mixing_energy).

 Polynomial regression analogy:
   Choice of degree (e.g. up to x^3)   -> ClusterSpace (cutoffs + allowed elements)
   [1, x, x^2, x^3] for a given x      -> cluster vector (82 numbers in this example) for a crystal
   Design matrix X (stacked rows)      -> StructureContainer
   Coefficients c_i from least-squares -> ECIs from ClusterExpansion fit
   Final model: y ≈ sum(c_i * x^i)     -> energy ≈ X @ ECI

 Cutoffs <-> polynomial degree: both set how expressive the feature space is.
 Looser cutoffs / higher degree = more features, richer model, more overfit risk.
 It's a modeling choice about the basis, NOT a filter on the data.

 We split since the basis is reusable — same ClusterSpace, swap datasets or add
 more properties (container can hold several targets per structure)

In [12]:
sc = StructureContainer(cluster_space=cs)
for row in db.select():
    sc.add_structure(structure=row.toatoms(),
                     user_tag=row.tag,
                     properties={'mixing_energy': row.mixing_energy})
display(sc)

,user_tag,natoms,formula,mixing_energy
0,Ag,1,Ag,0.000000
1,Pd,1,Pd,0.000000
2,AgPd_0002,2,AgPd,-0.039758
3,AgPd_0003,3,AgPd2,-0.028611
4,AgPd_0004,3,Ag2Pd,-0.048529
...,...,...,...,...
620,AgPd_0516,8,Ag4Pd4,-0.043282
621,AgPd_0517,8,Ag4Pd4,-0.040327
622,AgPd_0518,8,Ag4Pd4,-0.045361
623,AgPd_0519,8,Ag6Pd2,-0.046753


We can now use this information to fit the effective cluster interactions (ECIs) using the target mixing energies and therefore construct our cluster expansion. The goal here is to get best possible agreement with a set of training structures by using cross-validation. The final CE is then obtained by calling the train method as below.

In [15]:
opt = CrossValidationEstimator(
    fit_data=sc.get_fit_data(key='mixing_energy'), fit_method='ardr')
opt.validate()
opt.train()
print(opt)

============== CrossValidationEstimator ==============
seed                           : 42
fit_method                     : ardr
standardize                    : True
n_target_values                : 625
n_parameters                   : 82
n_nonzero_parameters           : 44
parameters_norm                : 0.07314093
target_values_std              : 0.01486413
rmse_train                     : 0.001632753
R2_train                       : 0.988
AIC                            : -7937.285
BIC                            : -7742.024
validation_method              : k-fold
n_splits                       : 10
rmse_train_final               : 0.001628285
rmse_validation                : 0.001889973
R2_validation                  : 0.9829637
shuffle                        : True


Here we see the number of nonzero parameters is almost half of the number of total parameters, so using the ardr lasso method for regularisation which helps with performance.